# Advanced Python Metaprogramming — Problems with Complete Solutions

This notebook is an expanded problem set based on three metaprogramming applications:

1. generating value-like classes with fields, properties, `__slots__`, equality, hashing, `repr`, and `str`;
2. implementing singleton behavior with metaclasses, especially the inheritance trap;
3. generating configuration classes and section classes from `.ini` data.

The exercises deliberately go beyond the introductory examples. They emphasize **correct class-creation timing, inheritance, immutability, constructor semantics, thread safety, dynamic class generation, typed configuration, lazy loading, metaclass composition, and `__prepare__`**.

## How to use this notebook

For each problem:

- read the **Problem** and try it before reading the solution code;
- run the **Solution** cell;
- run the **Verification / experiments** cell;
- inspect the assertions and modify the examples.

All code uses only the Python standard library.

> Best-practice note: metaclasses are powerful but expensive in complexity. Prefer ordinary functions, class decorators, descriptors, `__init_subclass__`, or `dataclasses` when they solve the problem cleanly. Use a metaclass when behavior genuinely belongs to **class creation or class calling**.

In [1]:
from __future__ import annotations

import configparser
import inspect
import keyword
import re
import tempfile
import threading
import time
from collections import OrderedDict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from types import SimpleNamespace
from typing import Any

def capture_error(exc_type, func, *args, **kwargs):
    """Run func and return the expected exception object."""
    try:
        func(*args, **kwargs)
    except exc_type as exc:
        return exc
    raise AssertionError(f"Expected {exc_type.__name__} to be raised")

## Problem 1 — Trace the full metaclass lifecycle

**Goal:** Prove, by instrumentation, which hooks run when a class is created and which run when an instance is created.

Create a metaclass that records:

- metaclass `__new__`;
- metaclass `__init__`;
- metaclass `__call__`;
- instance `__new__`;
- instance `__init__`.

Then answer:

1. Which hooks run at the `class Demo: ...` statement?
2. Which hook controls `Demo(...)`?
3. Where would singleton logic naturally live?

In [2]:
events = []

class TraceMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        events.append(("meta.__new__", name))
        return super().__new__(mcls, name, bases, namespace)

    def __init__(cls, name, bases, namespace, **kwargs):
        events.append(("meta.__init__", name))
        super().__init__(name, bases, namespace)

    def __call__(cls, *args, **kwargs):
        events.append(("meta.__call__", cls.__name__, args, kwargs))
        return super().__call__(*args, **kwargs)

class Demo(metaclass=TraceMeta):
    def __new__(cls, value):
        events.append(("instance.__new__", cls.__name__, value))
        return super().__new__(cls)

    def __init__(self, value):
        events.append(("instance.__init__", type(self).__name__, value))
        self.value = value

creation_events = list(events)
obj = Demo(10)

creation_events, events

([('meta.__new__', 'Demo'), ('meta.__init__', 'Demo')],
 [('meta.__new__', 'Demo'),
  ('meta.__init__', 'Demo'),
  ('meta.__call__', 'Demo', (10,), {}),
  ('instance.__new__', 'Demo', 10),
  ('instance.__init__', 'Demo', 10)])

In [3]:
assert [e[0] for e in creation_events] == ["meta.__new__", "meta.__init__"]
assert [e[0] for e in events[-3:]] == [
    "meta.__call__",
    "instance.__new__",
    "instance.__init__",
]
assert obj.value == 10

# Singleton logic naturally belongs in metaclass __call__ when the goal is to
# control what happens when the class object itself is called.

## Problem 2 — Fix late binding when generating properties

A metaclass loops over `_fields` and creates properties with lambdas. A common bug is that every generated getter closes over the **same loop variable**, so all properties read the final slot.

Implement both the broken version and the corrected version. Explain why the fix works.

In [4]:
class BrokenPropertyMeta(type):
    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)
        for field in namespace.get("_fields", ()):
            slot = f"_{field}"
            # BUG: "slot" is looked up when the lambda is called, not created.
            setattr(cls, field, property(lambda self: getattr(self, slot)))
        return cls

class BrokenPair(metaclass=BrokenPropertyMeta):
    _fields = ("x", "y")

    def __init__(self, x, y):
        self._x = x
        self._y = y

broken = BrokenPair(10, 20)
(broken.x, broken.y)

(20, 20)

In [5]:
class FixedPropertyMeta(type):
    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)
        for field in namespace.get("_fields", ()):
            slot = f"_{field}"

            # Default arguments are evaluated when the function is created.
            getter = lambda self, slot_name=slot: getattr(self, slot_name)
            setattr(cls, field, property(getter))
        return cls

class FixedPair(metaclass=FixedPropertyMeta):
    _fields = ("x", "y")

    def __init__(self, x, y):
        self._x = x
        self._y = y

fixed = FixedPair(10, 20)
assert (broken.x, broken.y) == (20, 20)
assert (fixed.x, fixed.y) == (10, 20)
(fixed.x, fixed.y)

(10, 20)

## Problem 3 — Create `__slots__` at the correct time

A subtle rule: assigning `__slots__` **after** the class has already been created does not retroactively change the instance layout.

Build a metaclass that:

- reads local `_fields`;
- injects the corresponding private slot names into the class namespace **before** calling `type.__new__`;
- creates read-only public properties;
- supports subclasses that add new fields;
- exposes `_all_fields` across the inheritance chain.

Verification should prove that instances have no `__dict__`.

In [6]:
class SlottedStructMeta(type):
    def __new__(mcls, name, bases, namespace):
        namespace = dict(namespace)
        local_fields = tuple(namespace.get("_fields", ()))

        if len(local_fields) != len(set(local_fields)):
            raise TypeError(f"{name}: duplicate field names")

        inherited_fields = tuple(
            field
            for base in bases
            for field in getattr(base, "_all_fields", ())
        )

        overlap = set(local_fields) & set(inherited_fields)
        if overlap:
            raise TypeError(f"{name}: fields already defined in a base: {sorted(overlap)}")

        namespace["_all_fields"] = inherited_fields + local_fields
        namespace["__slots__"] = tuple(f"_{field}" for field in local_fields)

        for field in local_fields:
            slot_name = f"_{field}"
            namespace[field] = property(
                lambda self, s=slot_name: getattr(self, s)
            )

        return super().__new__(mcls, name, bases, namespace)

class SlottedBase(metaclass=SlottedStructMeta):
    _fields = ()

class Point2D(SlottedBase):
    _fields = ("x", "y")

    def __init__(self, x, y):
        self._x = x
        self._y = y

class Point3D(Point2D):
    _fields = ("z",)

    def __init__(self, x, y, z):
        super().__init__(x, y)
        self._z = z

p2 = Point2D(1, 2)
p3 = Point3D(1, 2, 3)

assert Point2D._all_fields == ("x", "y")
assert Point3D._all_fields == ("x", "y", "z")
assert (p3.x, p3.y, p3.z) == (1, 2, 3)
assert not hasattr(p2, "__dict__")
assert not hasattr(p3, "__dict__")

(Point2D.__slots__, Point3D.__slots__, Point3D._all_fields)

(('_x', '_y'), ('_z',), ('x', 'y', 'z'))

## Problem 4 — Generate an immutable, hash-safe value class

If an object is hashable, values used by `__hash__` must not change while the object is in a set or dictionary.

Create a metaclass that generates:

- slots;
- an initializer;
- read-only public properties;
- exact-type equality;
- hashing;
- `repr`;
- a public constructor signature;
- immutability after initialization.

Avoid `exec`. Use `inspect.Signature` to bind arguments safely.

In [7]:
class FrozenValueMeta(type):
    def __new__(mcls, name, bases, namespace):
        namespace = dict(namespace)
        fields = tuple(namespace.get("_fields", ()))

        if len(fields) != len(set(fields)):
            raise TypeError(f"{name}: duplicate fields")
        for field in fields:
            if not field.isidentifier() or keyword.iskeyword(field):
                raise TypeError(f"{name}: invalid field name {field!r}")

        private_slots = tuple(f"_{f}" for f in fields)
        namespace["__slots__"] = private_slots + ("_frozen",)

        for field, private in zip(fields, private_slots):
            namespace[field] = property(
                lambda self, p=private: getattr(self, p)
            )

        params = [
            inspect.Parameter(f, inspect.Parameter.POSITIONAL_OR_KEYWORD)
            for f in fields
        ]
        sig = inspect.Signature(params)

        def __init__(self, *args, **kwargs):
            bound = sig.bind(*args, **kwargs)
            for field in fields:
                object.__setattr__(self, f"_{field}", bound.arguments[field])
            object.__setattr__(self, "_frozen", True)

        def __setattr__(self, attr, value):
            if getattr(self, "_frozen", False):
                raise AttributeError(f"{name} instances are immutable")
            object.__setattr__(self, attr, value)

        def __repr__(self):
            args = ", ".join(f"{f}={getattr(self, f)!r}" for f in fields)
            return f"{type(self).__name__}({args})"

        def __eq__(self, other):
            if type(self) is not type(other):
                return NotImplemented
            return all(getattr(self, f) == getattr(other, f) for f in fields)

        def __hash__(self):
            return hash(tuple(getattr(self, f) for f in fields))

        namespace.setdefault("__init__", __init__)
        namespace.setdefault("__setattr__", __setattr__)
        namespace.setdefault("__repr__", __repr__)
        namespace.setdefault("__eq__", __eq__)
        namespace.setdefault("__hash__", __hash__)

        cls = super().__new__(mcls, name, bases, namespace)
        cls.__signature__ = sig
        return cls

class Pixel(metaclass=FrozenValueMeta):
    _fields = ("x", "y", "label")

a = Pixel(2, 3, "hot")
b = Pixel(x=2, y=3, label="hot")

assert a == b
assert hash(a) == hash(b)
assert repr(a) == "Pixel(x=2, y=3, label='hot')"
assert str(inspect.signature(Pixel)) == "(x, y, label)"
assert not hasattr(a, "__dict__")

mutation_error = capture_error(AttributeError, setattr, a, "_x", 999)
(a, inspect.signature(Pixel), mutation_error)

(Pixel(x=2, y=3, label='hot'),
 <Signature (x, y, label)>,
 AttributeError('Pixel instances are immutable'))

## Problem 5 — Avoid rebuilding decorated classes when inheritance can do the job

A class decorator can receive a class and rebuild it with a different metaclass, but rebuilding is fragile:

- `__class__` cells and zero-argument `super()` can refer to the original class;
- metadata can be lost;
- descriptors may receive class-creation callbacks twice or at unexpected times.

A cleaner way to avoid repeatedly writing `metaclass=...` is often a **base class that already uses the metaclass**.

Create such a base class and show that subclasses automatically inherit the metaclass behavior.

In [8]:
class AutoPropsMeta(type):
    def __new__(mcls, name, bases, namespace):
        namespace = dict(namespace)
        fields = tuple(namespace.get("_fields", ()))
        namespace["__slots__"] = tuple(f"_{f}" for f in fields)

        for field in fields:
            private = f"_{field}"
            namespace[field] = property(
                lambda self, p=private: getattr(self, p)
            )

        return super().__new__(mcls, name, bases, namespace)

class StructBase(metaclass=AutoPropsMeta):
    _fields = ()

class Coordinate(StructBase):
    _fields = ("lat", "lon")

    def __init__(self, lat, lon):
        self._lat = lat
        self._lon = lon

c = Coordinate(42.6977, 23.3219)

assert isinstance(Coordinate, AutoPropsMeta)
assert (c.lat, c.lon) == (42.6977, 23.3219)
assert not hasattr(c, "__dict__")

type(Coordinate), Coordinate.__mro__

(__main__.AutoPropsMeta, (__main__.Coordinate, __main__.StructBase, object))

## Problem 6 — Repair the singleton inheritance trap

A naïve singleton implementation stores `existing_instance` on the target class. That breaks under inheritance because subclasses can inherit the parent's cached instance.

1. Reproduce the bug.
2. Fix it by keeping a dictionary on the **metaclass**, keyed by the exact class being instantiated.

In [9]:
class BadSingletonMeta(type):
    def __call__(cls, *args, **kwargs):
        if getattr(cls, "existing_instance", None) is None:
            cls.existing_instance = super().__call__(*args, **kwargs)
        return cls.existing_instance

class ParentBad(metaclass=BadSingletonMeta):
    pass

class ChildBad(ParentBad):
    pass

parent_bad = ParentBad()
child_bad = ChildBad()

assert child_bad is parent_bad
assert type(child_bad) is ParentBad
(type(parent_bad), type(child_bad), child_bad is parent_bad)

(__main__.ParentBad, __main__.ParentBad, True)

In [10]:
class SingletonMeta(type):
    _instances = {}

    def __call__(cls, *args, **kwargs):
        if cls not in SingletonMeta._instances:
            SingletonMeta._instances[cls] = super().__call__(*args, **kwargs)
        return SingletonMeta._instances[cls]

class ParentGood(metaclass=SingletonMeta):
    pass

class ChildGood(ParentGood):
    pass

p1, p2 = ParentGood(), ParentGood()
c1, c2 = ChildGood(), ChildGood()

assert p1 is p2
assert c1 is c2
assert p1 is not c1
assert type(c1) is ChildGood

(p1 is p2, c1 is c2, p1 is c1)

(True, True, False)

## Problem 7 — Define a constructor-argument policy for singletons

A singleton creates only one object, so what should happen here?

```python
Settings("prod")
Settings("dev")
```

Silently ignoring the second call's arguments is dangerous. Implement a strict policy:

- first call creates the instance and records constructor arguments;
- later calls with equivalent arguments return the same instance;
- later calls with different arguments raise `TypeError`.

This makes "first call wins" explicit instead of accidental.

In [11]:
class StrictSingletonMeta(type):
    _instances = {}
    _construction_calls = {}

    def __call__(cls, *args, **kwargs):
        call = (args, dict(kwargs))

        if cls not in cls._instances:
            instance = super().__call__(*args, **kwargs)
            cls._instances[cls] = instance
            cls._construction_calls[cls] = call
            return instance

        if cls._construction_calls[cls] != call:
            original = cls._construction_calls[cls]
            raise TypeError(
                f"{cls.__name__} is already initialized with {original}; "
                f"received incompatible call {call}"
            )

        return cls._instances[cls]

class Settings(metaclass=StrictSingletonMeta):
    def __init__(self, environment, timeout=5):
        self.environment = environment
        self.timeout = timeout

s1 = Settings("prod", timeout=10)
s2 = Settings("prod", timeout=10)
err = capture_error(TypeError, Settings, "dev", timeout=10)

assert s1 is s2
assert s1.environment == "prod"
(s1 is s2, str(err))

(True,
 "Settings is already initialized with (('prod',), {'timeout': 10}); received incompatible call (('dev',), {'timeout': 10})")

## Problem 8 — Make the singleton safe under concurrent construction

Without a lock, two threads can both observe "no instance yet" and both construct an object.

Implement a thread-safe singleton metaclass and stress it with a thread pool. The class initializer intentionally sleeps to widen the race window.

In [12]:
class ThreadSafeSingletonMeta(type):
    _instances = {}
    _lock = threading.RLock()

    def __call__(cls, *args, **kwargs):
        if cls in cls._instances:
            return cls._instances[cls]

        with cls._lock:
            if cls not in cls._instances:
                cls._instances[cls] = super().__call__(*args, **kwargs)
            return cls._instances[cls]

class ExpensiveClient(metaclass=ThreadSafeSingletonMeta):
    init_count = 0

    def __init__(self):
        type(self).init_count += 1
        time.sleep(0.01)

def build_client(_):
    return id(ExpensiveClient())

with ThreadPoolExecutor(max_workers=16) as pool:
    ids = list(pool.map(build_client, range(100)))

assert len(set(ids)) == 1
assert ExpensiveClient.init_count == 1
(len(set(ids)), ExpensiveClient.init_count)

(1, 1)

## Problem 9 — Generalize singleton into a keyed multiton

Sometimes "one instance for the entire class" is too restrictive. A **multiton** returns one instance per logical key.

Implement a metaclass where:

- the first positional argument is the cache key;
- equal keys reuse an object;
- different keys create different objects;
- cached objects are not reinitialized.

In [13]:
class MultitonMeta(type):
    _instances = {}
    _lock = threading.RLock()

    def __call__(cls, key, *args, **kwargs):
        try:
            hash(key)
        except TypeError as exc:
            raise TypeError("multiton keys must be hashable") from exc

        cache_key = (cls, key)
        with cls._lock:
            if cache_key not in cls._instances:
                cls._instances[cache_key] = super().__call__(key, *args, **kwargs)
            return cls._instances[cache_key]

class ServiceClient(metaclass=MultitonMeta):
    def __init__(self, endpoint, token):
        self.endpoint = endpoint
        self.token = token

a1 = ServiceClient("api-a", "token-A")
a2 = ServiceClient("api-a", "ignored-on-cache-hit")
b1 = ServiceClient("api-b", "token-B")

assert a1 is a2
assert a1 is not b1
assert a1.token == "token-A"
(a1 is a2, a1 is b1, a1.token, b1.token)

(True, False, 'token-A', 'token-B')

## Problem 10 — Generate config classes and section classes from `.ini`

Build a robust version of the dynamic configuration idea.

Requirements:

- the metaclass receives `config_path=...`;
- the file must exist and must actually be read;
- each INI section becomes a dynamically generated **class**;
- each key/value pair becomes a class attribute;
- the overall config class keeps its original base classes;
- documentation strings identify the source.

The test writes temporary INI files, so the notebook is self-contained.

In [14]:
_tmp = tempfile.TemporaryDirectory()
CONFIG_DIR = Path(_tmp.name)

dev_ini = CONFIG_DIR / "dev.ini"
prod_ini = CONFIG_DIR / "prod.ini"

dev_ini.write_text(
    "[Database]\n"
    "db_host=dev.internal\n"
    "db_name=app\n"
    "\n"
    "[Server]\n"
    "port=3000\n",
    encoding="utf-8",
)

prod_ini.write_text(
    "[Database]\n"
    "db_host=prod.internal\n"
    "db_name=app\n"
    "\n"
    "[Server]\n"
    "port=8080\n",
    encoding="utf-8",
)

class SectionType(type):
    def __new__(mcls, name, bases, namespace, *, section_name, items):
        namespace = dict(namespace)
        namespace["__doc__"] = f"Configuration values for [{section_name}]"
        namespace["section_name"] = section_name
        namespace.update(dict(items))
        return super().__new__(mcls, name, bases, namespace)

class ConfigType(type):
    def __new__(mcls, name, bases, namespace, *, config_path):
        namespace = dict(namespace)
        path = Path(config_path)

        parser = configparser.ConfigParser()
        loaded = parser.read(path, encoding="utf-8")
        if not loaded:
            raise FileNotFoundError(path)

        namespace["__doc__"] = f"Configuration generated from {path.name}"
        namespace["config_path"] = path

        for section_name in parser.sections():
            section_class_name = f"{section_name}Section"
            section_cls = SectionType(
                section_class_name,
                (object,),
                {},
                section_name=section_name,
                items=parser[section_name].items(),
            )
            namespace[section_name.casefold()] = section_cls

        return super().__new__(mcls, name, bases, namespace)

class DevConfig(metaclass=ConfigType, config_path=dev_ini):
    pass

class ProdConfig(metaclass=ConfigType, config_path=prod_ini):
    pass

assert DevConfig.database.db_host == "dev.internal"
assert ProdConfig.database.db_host == "prod.internal"
assert DevConfig.server.port == "3000"
assert "dev.ini" in DevConfig.__doc__

(
    DevConfig.database.db_host,
    ProdConfig.server.port,
    DevConfig.__doc__,
    DevConfig.database.__doc__,
)

('dev.internal',
 '8080',
 'Configuration generated from dev.ini',
 'Configuration values for [Database]')

## Problem 11 — Find and fix a shadowed-`bases` metaclass bug

A dangerous implementation detail is easy to miss:

```python
def __new__(mcls, name, bases, namespace, ...):
    ...
    for section in ...:
        bases = (object,)   # accidental overwrite
        ...
    return super().__new__(mcls, name, bases, namespace)
```

The local variable intended for dynamically created section classes overwrites the **real bases of the config class**.

Reproduce the failure with a marker base class, then fix it by using a different local name.

In [15]:
class Marker:
    marker = "inherited"

class BuggyConfigType(type):
    def __new__(mcls, name, bases, namespace, *, config_path):
        parser = configparser.ConfigParser()
        parser.read(config_path, encoding="utf-8")
        namespace = dict(namespace)

        for section_name in parser.sections():
            bases = (object,)
            section_cls = type(f"{section_name}Section", bases, {})
            namespace[section_name.casefold()] = section_cls

        return super().__new__(mcls, name, bases, namespace)

class BrokenConfig(Marker, metaclass=BuggyConfigType, config_path=dev_ini):
    pass

assert not issubclass(BrokenConfig, Marker)
BrokenConfig.__mro__

(__main__.BrokenConfig, object)

In [16]:
class FixedConfigType(type):
    def __new__(mcls, name, bases, namespace, *, config_path):
        parser = configparser.ConfigParser()
        parser.read(config_path, encoding="utf-8")
        namespace = dict(namespace)

        for section_name in parser.sections():
            section_bases = (object,)
            section_cls = type(
                f"{section_name}Section",
                section_bases,
                {},
            )
            namespace[section_name.casefold()] = section_cls

        return super().__new__(mcls, name, bases, namespace)

class FixedConfig(Marker, metaclass=FixedConfigType, config_path=dev_ini):
    pass

assert issubclass(FixedConfig, Marker)
assert FixedConfig.marker == "inherited"
FixedConfig.__mro__

(__main__.FixedConfig, __main__.Marker, object)

## Problem 12 — Add typed conversion and strict schema validation

`configparser` returns strings. Real applications often need integers, booleans, paths, or custom types.

Create a config metaclass that accepts a schema such as:

```python
{
    "Server": {"port": int, "debug": parse_bool},
    "Database": {"pool_size": int},
}
```

Requirements:

- required sections and keys must exist;
- converters run during class creation;
- conversion failures produce contextual errors;
- generated section classes contain already-converted values.

In [17]:
typed_ini = CONFIG_DIR / "typed.ini"
typed_ini.write_text(
    "[Server]\n"
    "port=9000\n"
    "debug=yes\n"
    "\n"
    "[Database]\n"
    "pool_size=12\n",
    encoding="utf-8",
)

def parse_bool(text):
    normalized = text.strip().casefold()
    if normalized in {"1", "true", "yes", "on"}:
        return True
    if normalized in {"0", "false", "no", "off"}:
        return False
    raise ValueError(f"invalid boolean {text!r}")

class TypedConfigMeta(type):
    def __new__(mcls, name, bases, namespace, *, config_path, schema):
        path = Path(config_path)
        parser = configparser.ConfigParser()
        loaded = parser.read(path, encoding="utf-8")
        if not loaded:
            raise FileNotFoundError(path)

        namespace = dict(namespace)
        namespace["config_path"] = path

        for section_name, key_schema in schema.items():
            if not parser.has_section(section_name):
                raise KeyError(f"{path.name}: missing section [{section_name}]")

            converted = {}
            for key, converter in key_schema.items():
                if not parser.has_option(section_name, key):
                    raise KeyError(
                        f"{path.name}: [{section_name}] missing key {key!r}"
                    )

                raw = parser.get(section_name, key)
                try:
                    converted[key] = converter(raw)
                except Exception as exc:
                    raise ValueError(
                        f"{path.name}: cannot convert "
                        f"[{section_name}] {key}={raw!r}"
                    ) from exc

            section_cls = type(f"{section_name}Section", (), converted)
            namespace[section_name.casefold()] = section_cls

        return super().__new__(mcls, name, bases, namespace)

SCHEMA = {
    "Server": {"port": int, "debug": parse_bool},
    "Database": {"pool_size": int},
}

class AppConfig(metaclass=TypedConfigMeta, config_path=typed_ini, schema=SCHEMA):
    pass

assert AppConfig.server.port == 9000
assert AppConfig.server.debug is True
assert AppConfig.database.pool_size == 12
assert isinstance(AppConfig.server.port, int)

(
    AppConfig.server.port,
    type(AppConfig.server.port),
    AppConfig.server.debug,
    AppConfig.database.pool_size,
)

(9000, int, True, 12)

## Problem 13 — Sanitize dynamic names and detect collisions

External configuration names are data, not necessarily valid Python identifiers.

Examples:

- `api-key`
- `2fa`
- `class`

Write a sanitizer for dot-notation attributes and **reject collisions** such as `api-key` and `api key`, which both normalize to the same identifier.

Do not silently overwrite one value with another.

In [18]:
_non_identifier = re.compile(r"\W", flags=re.UNICODE)

def safe_identifier(raw):
    name = _non_identifier.sub("_", raw.strip())
    if not name:
        raise ValueError(f"cannot derive identifier from {raw!r}")
    if name[0].isdigit():
        name = "_" + name
    if keyword.iskeyword(name):
        name += "_"
    if not name.isidentifier():
        raise ValueError(f"cannot safely normalize {raw!r}")
    return name

def normalize_items(items):
    normalized = {}
    origins = {}

    for raw_name, value in items:
        name = safe_identifier(raw_name)
        if name in normalized:
            raise ValueError(
                f"name collision: {origins[name]!r} and {raw_name!r} -> {name!r}"
            )
        normalized[name] = value
        origins[name] = raw_name

    return normalized

safe = normalize_items([
    ("api-key", "secret"),
    ("2fa", "enabled"),
    ("class", "gold"),
])

SafeSection = type("SafeSection", (), safe)

assert SafeSection.api_key == "secret"
assert SafeSection._2fa == "enabled"
assert SafeSection.class_ == "gold"

collision = capture_error(
    ValueError,
    normalize_items,
    [("api-key", 1), ("api key", 2)],
)

(safe, str(collision))

({'api_key': 'secret', '_2fa': 'enabled', 'class_': 'gold'},
 "name collision: 'api-key' and 'api key' -> 'api_key'")

## Problem 14 — Delay configuration I/O with metaclass `__getattr__`

Reading files during class creation means I/O happens at import time. That can be undesirable for:

- command-line tools that do not always need the config;
- test discovery;
- optional features;
- applications where files appear later.

Implement a **lazy config metaclass**:

- the class is cheap to define;
- the first unknown section attribute triggers loading;
- all sections are cached as class attributes;
- subsequent access does not reread the file.

In [19]:
class LazyConfigMeta(type):
    def __getattr__(cls, attr):
        if not type.__getattribute__(cls, "_loaded"):
            cls._load()

        try:
            return type.__getattribute__(cls, attr)
        except AttributeError:
            raise AttributeError(
                f"{cls.__name__} has no config section {attr!r}"
            ) from None

class LazyConfig(metaclass=LazyConfigMeta):
    _config_path = dev_ini
    _loaded = False
    _load_count = 0

    @classmethod
    def _load(cls):
        parser = configparser.ConfigParser()
        loaded = parser.read(cls._config_path, encoding="utf-8")
        if not loaded:
            raise FileNotFoundError(cls._config_path)

        for section_name in parser.sections():
            section = SimpleNamespace(**dict(parser[section_name].items()))
            setattr(cls, section_name.casefold(), section)

        cls._loaded = True
        cls._load_count += 1

assert LazyConfig._loaded is False

first = LazyConfig.database.db_host
second = LazyConfig.server.port
third = LazyConfig.database.db_name

assert first == "dev.internal"
assert second == "3000"
assert third == "app"
assert LazyConfig._load_count == 1

(first, second, third, LazyConfig._load_count)

('dev.internal', '3000', 'app', 1)

## Problem 15 — Resolve a metaclass conflict by composing metaclasses

Suppose one class needs both:

- immutable value-object generation from `FrozenValueMeta`;
- strict singleton construction from `StrictSingletonMeta`.

Python allows only one metaclass for a class, so compose a metaclass that subclasses both. Confirm that:

- class creation goes through the value metaclass;
- class calls go through singleton `__call__`;
- repeated equivalent construction returns the same object;
- incompatible construction is rejected.

In [20]:
class FrozenSingletonMeta(FrozenValueMeta, StrictSingletonMeta):
    pass

class GlobalConfig(metaclass=FrozenSingletonMeta):
    _fields = ("environment", "region")

g1 = GlobalConfig("prod", "eu")
g2 = GlobalConfig("prod", "eu")
g3_error = capture_error(TypeError, GlobalConfig, "dev", "eu")

assert g1 is g2
assert repr(g1) == "GlobalConfig(environment='prod', region='eu')"
assert hash(g1) == hash(g2)

(g1 is g2, g1, str(g3_error), FrozenSingletonMeta.__mro__)

(True,
 GlobalConfig(environment='prod', region='eu'),
 "GlobalConfig is already initialized with (('prod', 'eu'), {}); received incompatible call (('dev', 'eu'), {})",
 (__main__.FrozenSingletonMeta,
  __main__.FrozenValueMeta,
  __main__.StrictSingletonMeta,
  type,
  object))

## Problem 16 — Use `__prepare__` for a declarative typed schema

`type.__prepare__` controls the mapping used while Python executes a class body.

Build a small declarative record system:

```python
class User(metaclass=SchemaMeta):
    name = Field(str)
    age = Field(int)
```

Requirements:

- preserve declaration order;
- each `Field` validates assignments;
- storage uses private slots;
- constructor arguments follow declaration order;
- `repr` is generated;
- duplicate assignment to the same name in the class body is rejected.

This is a more advanced example because the metaclass participates **before**, **during**, and **after** execution of the class body.

In [21]:
class DuplicateCheckingDict(OrderedDict):
    _allowed_reassignments = {"__module__", "__qualname__", "__annotations__"}

    def __setitem__(self, key, value):
        if key in self and key not in self._allowed_reassignments:
            raise TypeError(f"duplicate class-body name: {key!r}")
        super().__setitem__(key, value)

class Field:
    def __init__(self, expected_type):
        self.expected_type = expected_type
        self.name = None
        self.storage_name = None

    def __set_name__(self, owner, name):
        self.name = name
        self.storage_name = f"_{name}"

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return getattr(instance, self.storage_name)

    def __set__(self, instance, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(
                f"{self.name} must be {self.expected_type.__name__}; "
                f"got {type(value).__name__}"
            )
        object.__setattr__(instance, self.storage_name, value)

class SchemaMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        return DuplicateCheckingDict()

    def __new__(mcls, name, bases, namespace):
        fields = [
            (field_name, value)
            for field_name, value in namespace.items()
            if isinstance(value, Field)
        ]
        field_names = tuple(field_name for field_name, _ in fields)

        namespace["__slots__"] = tuple(f"_{field_name}" for field_name in field_names)
        namespace["_field_names"] = field_names

        def __init__(self, *args, **kwargs):
            sig = inspect.Signature([
                inspect.Parameter(f, inspect.Parameter.POSITIONAL_OR_KEYWORD)
                for f in field_names
            ])
            bound = sig.bind(*args, **kwargs)
            for field_name in field_names:
                setattr(self, field_name, bound.arguments[field_name])

        def __repr__(self):
            values = ", ".join(
                f"{f}={getattr(self, f)!r}" for f in field_names
            )
            return f"{type(self).__name__}({values})"

        namespace.setdefault("__init__", __init__)
        namespace.setdefault("__repr__", __repr__)

        return super().__new__(mcls, name, bases, dict(namespace))

class User(metaclass=SchemaMeta):
    name = Field(str)
    age = Field(int)
    active = Field(bool)

u = User("Ada", 37, True)
assert repr(u) == "User(name='Ada', age=37, active=True)"
assert User._field_names == ("name", "age", "active")
assert not hasattr(u, "__dict__")

type_error = capture_error(TypeError, setattr, u, "age", "old")

duplicate_error = capture_error(
    TypeError,
    exec,
    """
class BadSchema(metaclass=SchemaMeta):
    x = Field(int)
    x = Field(str)
""",
    globals(),
    {},
)

(u, str(type_error), str(duplicate_error))

(User(name='Ada', age=37, active=True),
 'age must be int; got str',
 "duplicate class-body name: 'x'")

# Capstone — Design review and decision exercise

You are asked to design a production configuration object with these requirements:

- one logical configuration per environment;
- typed values;
- cheap imports;
- testable file I/O;
- clear introspection;
- no accidental mutation.

### Questions

1. Would you use a strict singleton, a keyed multiton, or explicit dependency injection?
2. Would you parse files in metaclass `__new__` or lazily?
3. Which data should live on the class versus instances?
4. How would you make errors include file, section, and key context?
5. Which parts are better served by `dataclasses`, descriptors, or plain functions rather than a metaclass?

### Reference solution

A strong production design would usually **separate concerns**:

- use a plain loader function for I/O and conversion;
- represent parsed values with immutable dataclasses or frozen value objects;
- cache by environment only if measurements justify caching;
- inject the resulting config into consumers rather than making global access mandatory;
- reserve a metaclass for class-definition-time validation or declarative schema behavior.

The earlier exercises show that metaclasses *can* implement all of these behaviors; the design question is whether they *should*.

# Additional practice problems

These are intentionally left as extensions after the fully solved set above.

1. Add default values to `FrozenValueMeta` while keeping an accurate `inspect.signature`.
2. Support keyword-only fields.
3. Make `ThreadSafeSingletonMeta` use one lock per target class instead of one global lock.
4. Replace the strong singleton cache with `weakref.WeakValueDictionary` and discuss when the object may disappear.
5. Add environment-variable overrides to `TypedConfigMeta`.
6. Add nested section names such as `[database.replica]`.
7. Generate `__match_args__` so value objects work naturally with structural pattern matching.
8. Add a `to_dict()` method generated by the metaclass.
9. Detect mutable field values before allowing `__hash__`.
10. Benchmark metaclass-generated records against `dataclasses.dataclass(frozen=True, slots=True)`.

## Final checklist

When writing metaclasses, verify:

- **timing:** does the change need to happen before `type.__new__`?
- **inheritance:** are caches and generated fields isolated per class?
- **cooperation:** do overrides call `super()` so metaclasses can compose?
- **signatures:** do generated constructors remain inspectable?
- **hash safety:** can hashed values mutate?
- **thread safety:** can class calls race?
- **I/O timing:** is import-time work acceptable?
- **naming:** can external data become invalid or colliding identifiers?
- **simpler alternatives:** would a dataclass, decorator, descriptor, function, or `__init_subclass__` be clearer?